In [1]:

import torch, torchvision
import torch.nn as nn, torch.optim as optim
import torchvision.transforms as T 
from torch.utils.data import DataLoader, Subset

In [4]:
t = T.Compose([
    T.Resize((224,224)),
    T.Grayscale(3),
    T.ToTensor(),
    T.Normalize((.5,.5,.5),(.5,.5,.5))
])

trd = DataLoader(Subset(torchvision.datasets.MNIST('./data',True,transform=t,download=True),range(2000)),64,True)
ted = DataLoader(Subset(torchvision.datasets.MNIST('./data',False,transform=t,download=True),range(500)),64,)

In [7]:
class AlexNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.feats = nn.Sequential(
            nn.Conv2d(3,64,11,4,2), nn.ReLU(), nn.MaxPool2d(3,2),
            nn.Conv2d(64,192,5,padding=2), nn.ReLU(), nn.MaxPool2d(3,2),
            nn.Conv2d(192,384,3,padding=1), nn.ReLU(),
            nn.Conv2d(384,256,3,padding=1), nn.ReLU(),
            nn.Conv2d(256,256,3,padding=1), nn.ReLU(), nn.MaxPool2d(3,2),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(), nn.Linear(9216,4096),
            nn.Dropout(), nn.Linear(4096,4096), nn.ReLU(),
            nn.Linear(4096,10)
        )
    def forward(self,x):
        x = self.feats(x)
        x = x.view(x.size(0),-1)
        x = self.classifier(x)
        return x

In [10]:
model = AlexNet()
opt = optim.Adam(model.parameters(),1e-3)
crit = nn.CrossEntropyLoss()

for e in range(1):
    model.train()
    rl=0
    for i,(x,y) in enumerate(trd):
        opt.zero_grad()
        o = model(x)
        l = crit(o,y)
        l.backward()
        opt.step()
        rl += l.item()
        if i%100==0:
            print(f'Loss {rl:.2f} Batch {i+1} ')
            rl=0
    
    model.eval()
    c=n=0
    for x,y in ted:
        c += (model(x).argmax(1)==y).sum().item()
        n += len(y)
    print(f'Acc : {c/n*100:.2f}%')

Loss 2.30 Batch 1 
Acc : 9.80%
